# Tools

A model's capabilities is extended with tools. Tools can be built into the API or accessed through functions and Model Context Protocol (MCP) services.

Tools include:

- Image generation.
- Web search.
- Function calling.
- Remote MCP Servers.
- File search.
- Code interpreter.
- Computer use.

# Function Calling

In this notebook we use function calling to illustrate tools and their usage. 

A useful resource is OpenAI's [Function Calling documentation for the responses API](https://platform.openai.com/docs/guides/function-calling?api-mode=responses).

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.secrets

In [2]:
from openai import OpenAI
import json
import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

# Tool Calling Flow

The idea behind using tools is that a model will request to use a tool. When a model examines a prompt, it may determine that to follow the instruction it should use a tool. It then generates a special type of output, which we will catch and call a specific tool. 

The logic flows as follows:


1. Make a request to the model with tools it could call
2. Receive a tool call from the model
3. Execute code on the application side with input from the tool call
4. Make a second request to the model with the tool output
5. Receive a final response from the model (or more tool calls)


<img src="./img/05_function-calling-diagram-steps.png" height=700>

# Make a request to the model with tools it could call

The tool definition includes the function name, its parameters, and additional metadata. 

In [ ]:
import requests

tools = [
    {
        "type": "function",
        "name": "get_basketball_players_list",
        "description": "Retrieve the list of players for the specified basketball team.",
        "parameters": {
            "type": "object",
            "properties": {
                "team_name": {
                    "type": "string",
                    "description": "The name of the basketball team.",
                }
            },
            "required": ["team_name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "get_basketball_team_ranking",
        "description": "Retrieve the current ranking of an NBA basketball team in 2026, including their win and loss counts.",
        "parameters": {
            "type": "object",
            "properties": {
                "team_name": {
                    "type": "string",
                    "description": "The name of the basketball team.",
                }
            },
            "required": ["team_name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]


all_teams = [
    {
        "id": 1,
        "full_name": "atlanta hawks"
    },
    {
        "id": 4,
        "full_name": "charlotte hornets"
    }
]



def get_basketball_players_list(team_name: str = "atlanta hawks") -> str:
    """
    Retrieve the list of players for the specified team in the National Basketball Association.
    """

    team_id = 1
    for team in all_teams:
        if team_name.lower() in team["full_name"].lower():
            team_id = team["id"]
            break

    url = f"https://api.balldontlie.io/v1/players?team_ids[]={team_id}"
    response = requests.get(url, headers={'Authorization': '6712f035-bd42-4cf8-88e4-fc7bffdbf25e'})
    return response.text


We will need to retain some "memory" of the conversation. In this case, we will retain the message history and send it as context in our interaction with the model

In [55]:
input_list = [
    {"role": "user", "content": "Which players are on the Charlotte Hornets?"}
]

We prompt the model with the initial input.

In [56]:
response = client.responses.create(
    model="gpt-4o-mini",
    tools=tools,
    input=input_list,
)

In [57]:
print(tools)

[{'type': 'function', 'name': 'get_basketball_players_list', 'description': 'Retrieve the list of players for the specified basketball team.', 'parameters': {'type': 'object', 'properties': {'team_name': {'type': 'string', 'description': 'The name of the basketball team.'}}, 'required': ['team_name'], 'additionalProperties': False}, 'strict': True}]


# Execute code on the application side with input from the tool call


Examine the response output. Notice that we find a "reasoning item" and a "function tool call".

In [58]:
response.output

[ResponseFunctionToolCall(arguments='{"team_name":"Charlotte Hornets"}', call_id='call_5jXa9U07RukRidCK6QTx8JHU', name='get_basketball_players_list', type='function_call', id='fc_03e65334953d41750069a36f8cab7481a19509e32552e61c93', status='completed')]

The function call item indicates that the model is requesting to run the function call: `get_horoscope(zodiac_sign='Aquarius')`.

In [59]:
response.output[0].model_dump()

{'arguments': '{"team_name":"Charlotte Hornets"}',
 'call_id': 'call_5jXa9U07RukRidCK6QTx8JHU',
 'name': 'get_basketball_players_list',
 'type': 'function_call',
 'id': 'fc_03e65334953d41750069a36f8cab7481a19509e32552e61c93',
 'status': 'completed'}

In the code below:

+ Execute the function logic for get_horoscope.
+ Add the function output to the input_list.

In [60]:
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_basketball_players_list":
            # Execute the function logic for get_basketball_players_list
            resp = get_basketball_players_list(**json.loads(item.arguments))
            
            # Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "players_data": resp
                })
            })

# Make a second request to the model with the tool output


Examine the final input_list which is the context of model call.

In [61]:
print(input_list)

[{'role': 'user', 'content': 'Which players are on the Charlotte Hornets?'}, ResponseFunctionToolCall(arguments='{"team_name":"Charlotte Hornets"}', call_id='call_5jXa9U07RukRidCK6QTx8JHU', name='get_basketball_players_list', type='function_call', id='fc_03e65334953d41750069a36f8cab7481a19509e32552e61c93', status='completed'), {'type': 'function_call_output', 'call_id': 'call_5jXa9U07RukRidCK6QTx8JHU', 'output': '{"players_data": "{\\"data\\":[{\\"id\\":44,\\"first_name\\":\\"Davis\\",\\"last_name\\":\\"Bertans\\",\\"position\\":\\"F\\",\\"height\\":\\"6-10\\",\\"weight\\":\\"225\\",\\"jersey_number\\":\\"9\\",\\"college\\":\\"Baskonia\\",\\"country\\":\\"Latvia\\",\\"draft_year\\":2011,\\"draft_round\\":2,\\"draft_number\\":42,\\"team\\":{\\"id\\":4,\\"conference\\":\\"East\\",\\"division\\":\\"Southeast\\",\\"city\\":\\"Charlotte\\",\\"name\\":\\"Hornets\\",\\"full_name\\":\\"Charlotte Hornets\\",\\"abbreviation\\":\\"CHA\\"}},{\\"id\\":62,\\"first_name\\":\\"Miles\\",\\"last_name\\"

In [62]:
response = client.responses.create(
    model="gpt-4o-mini",
    instructions="Respond only with the information generated by a tool.",
    tools=tools,
    input=input_list,
)


# Receive a final response from the model (or more tool calls)

In [63]:
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

{
  "id": "resp_03e65334953d41750069a36fa67b1481a1a9a873f8dc612e6e",
  "created_at": 1772318630.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with the information generated by a tool.",
  "metadata": {},
  "model": "gpt-4o-mini-2024-07-18",
  "object": "response",
  "output": [
    {
      "id": "msg_03e65334953d41750069a36fa71b7881a1b640dcbb7a135747",
      "content": [
        {
          "annotations": [],
          "text": "Here are the players on the Charlotte Hornets:\n\n1. **Davis Bertans**\n   - Position: Forward\n   - Height: 6'10\"\n   - Weight: 225 lbs\n   - Jersey Number: 9\n   - College: Baskonia\n   - Country: Latvia\n\n2. **Miles Bridges**\n   - Position: Forward\n   - Height: 6'7\"\n   - Weight: 225 lbs\n   - Jersey Number: 0\n   - College: Michigan State\n   - Country: USA\n\n3. **Joe Chealey**\n   - Position: Guard\n   - Height: 6'3\"\n   - Weight: 190 lbs\n   - Jersey Number: 31\n   - College: College of Charleston\n   - Country: U